# 07. Gold: assign clusters and write a sorted table

Gives every review its cluster_id and writes a table sorted by it, so Iceberg can skip whole files when you search.

## What to look at

- **Files per month** and the **cluster_id min..max inside each file**: narrow ranges are what make skipping possible.
- **Pruning preview**: for 1, 3 and 5 random clusters, how many of the data files Iceberg would have to open.
- **Sample reviews** from the biggest clusters share a theme.

In [ ]:
import sys, pathlib
_nb = next(p for p in (pathlib.Path.cwd(), pathlib.Path.cwd() / "notebooks") if (p / "_setup.py").exists())
sys.path.insert(0, str(_nb))
from _setup import *   # puts src/ on the path, loads .env, gives run_script / get_spark / sql / minio helpers
print("SAMPLE_ROWS =", common.SAMPLE_ROWS, "(None = all rows)")

In [ ]:
run_script("07_gold_write.py")

In [ ]:
spark = get_spark("nb07")
gold = load_script("07_gold_write.py")
nfiles, nbytes, avg = gold.gold_file_stats(spark)
pd.DataFrame([("data files", nfiles), ("total MB", round(nbytes / 1e6, 1)), ("average KB per file", round(avg / 1e3)),
              ("table sort order", spark.sql(f"SHOW TBLPROPERTIES {T_GOLD}").where("key = 'sort-order'").first().value)],
             columns=["", "value"])

In [ ]:
sql(spark, f"""SELECT add_months(DATE'1970-01-01', partition.purchase_ts_month) AS month, record_count AS rows,
                      readable_metrics.cluster_id.lower_bound AS cluster_min, readable_metrics.cluster_id.upper_bound AS cluster_max
               FROM {T_GOLD}.files ORDER BY month, cluster_min LIMIT 15""")

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
k = spark.table(T_CENTROIDS).count()
rows = []
for n in (1, 3, 5):
    probe = [int(c) for c in rng.choice(k, size=n, replace=False)]
    hit, total = gold.files_for_clusters(spark, probe)
    rows.append((n, probe, hit, total, f"{100 * (1 - hit / total):.0f}% skipped"))
pd.DataFrame(rows, columns=["clusters probed", "cluster ids", "files to open", "total files", "result"])

In [ ]:
from pyspark.sql import functions as F
g = spark.table(T_GOLD)
top = [r.cluster_id for r in g.groupBy("cluster_id").count().orderBy(F.desc("count")).limit(3).collect()]
pd.concat([g.where(F.col("cluster_id") == c).select("cluster_id", "review_text").limit(2).toPandas() for c in top])

In [ ]:
# OPTIONAL maintenance demo (writes the table 3 times, compacts it, then restores the normal table). Off by default.
DEMO_COMPACTION = False
if DEMO_COMPACTION:
    stop_spark()
    run_script("07_gold_write.py", "--fragment", "3", "--compact")   # shows before/after file counts in its log
    run_script("07_gold_write.py")                                   # restore the normal single-write table

In [ ]:
# Cleanup. Skip this cell while you are showing the Spark UI (http://localhost:4040): it closes with the session.
stop_spark()

## Open MinIO now

http://localhost:9001 (login `minioadmin` / `minioadmin`), bucket **warehouse**.

Open `gold/review_embeddings/data/`. There is **one folder per month** (`purchase_ts_month=2017-01`, ...) and inside each **several small Parquet files** (a few hundred KB or less). Each file holds a narrow range of clusters; that is what makes searching one cluster cheap. `metadata/` holds the table metadata, manifest list and manifests.